In [1]:
from sys import platform
from pathlib import Path
import os
from requests_html import HTMLSession
import wget
import imageio.v3 as iio
from tqdm import tqdm

#### System Paths

In [2]:
if platform == 'linux':
    home_dir = os.path.expanduser('~')
elif platform == 'win32':
    home_dir = os.path.expandvars(r'%HOMEDRIVE%%HOMEPATH%\OneDrive') #Stupid OneDrive
else:
    raise ValueError("Cannot set home directory for this OS.")
print(home_dir)

/home/monorhesus


#### Body sex

In [3]:
sex = 'Male' # Female

#### Output directories

In [4]:
img_dir = os.path.join('Pictures', 'visible-human', sex.lower(), 'raw')
img_dir

'Pictures/visible-human/male/raw'

In [6]:
wk_dir = os.path.join(home_dir, img_dir)
wd_path = Path(wk_dir)
wd_path.mkdir(parents=True, exist_ok=True)
print(f'Working dir: {wk_dir}')

Working dir: /home/monorhesus/Pictures/visible-human/male/raw


#### Get top-level index URLs

In [7]:
index_url = f'https://data.lhncbc.nlm.nih.gov/public/Visible-Human/{sex}-Images/PNG_format/index.html'
print(f'Getting {sex} images.')

Getting Male images.


In [8]:
session = HTMLSession()

In [9]:
i = session.get(index_url)
urls_all = i.html.absolute_links
index_urls = [u for u in urls_all if 'radiological' not in u]
index_urls

['https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/head/index.html',
 'https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/abdomen/index.html',
 'https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/pelvis/index.html',
 'https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/legs/index.html',
 'https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/thorax/index.html',
 'https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/thighs/index.html']

#### Get images

In [11]:
for url in index_urls:
    r = session.get(url)
    img_urls = list(r.html.absolute_links)
    img_urls[:5]
    
    # Create body part directory
    body_part = list(set([u.split('/')[-2] for u in img_urls]))
    assert len(body_part) == 1, 'Error parsing body part'
    body_dirname = body_part[0]
    body_dir = os.path.join(wk_dir, body_dirname)
    mkbody_dir = Path(body_dir)
    mkbody_dir.mkdir(parents=True, exist_ok=True)
    print(f'{body_part[0].upper()}')
    
    for img_url in tqdm(img_urls, desc='Download Progress', total=len(img_urls)):
        #TODO: Concurrent
        if img_url.endswith('.png'):
            
            # Avoid wget duplicates
            img_name = img_url.split('/')[-1]
            filename = rf'{body_dir}\{img_name}'
            if os.path.exists(filename):
                # print(f'{filename} exists, skipping')
                pass
            else:
                # Download image
                wget.download(img_url, out=body_dir, bar=None)
                # print(f'\nDownloaded {body_dirname}/{img_name}')

HEAD


Download Progress: 100%|██████████| 377/377 [04:04<00:00,  1.54it/s]


ABDOMEN


Download Progress: 100%|██████████| 543/543 [05:20<00:00,  1.70it/s]


PELVIS


...

THORAX


Download Progress: 100%|██████████| 409/409 [05:08<00:00,  1.32it/s]


THIGHS


Download Progress:  24%|██▍       | 165/680 [02:27<07:38,  1.12it/s]


KeyboardInterrupt: 

#### Animate segmentation
Takes a while and generates a 2.6Gb gif, see other script for other methods.

In [ ]:
sorted_imgs = {}
for root, dirs, files in os.walk(wk_dir):
    for file in files:
        k = int(''.join([c for c in file if c.isdigit()]))
        v = os.path.join(root, file)
        sorted_imgs[k] = v
sorted_imgs = {k: sorted_imgs[k] for k in sorted(sorted_imgs)} # Sort images 
[sorted_imgs[k] for k in list(sorted_imgs.keys())[:10]] # Verify proper order for axial segmentation

In [ ]:
filenames = [sorted_imgs[k] for k in sorted_imgs] 
images = [iio.imread(f) for f in filenames]
iio.imwrite(f'{wk_dir}\\gif-visiblehuman-axial.gif', images, duration=25, loop=0)